# channel-list-reverse-build — faded example 2: Iterate reversed channel pairs to build generator ConvTranspose blocks

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `channel-list-reverse-build`. Running the beacon reports progress on the `GAN: channel-list reverse build` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: channel-list reverse build` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`channel-list-reverse-build`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "channel-list-reverse-build"
DD_SUBTOPIC = "GAN: channel-list reverse build"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The generator's `nn.Sequential` is built by reversing the discriminator's `hidden_channels` and iterating consecutive `(in_c, out_c)` pairs, appending a `ConvTranspose2d` + `BatchNorm2d` + `ReLU` per pair. The ConvTranspose uses `kernel_size=4, stride=2, padding=1, bias=False` so each block doubles spatial size and the following BatchNorm absorbs the bias.

## Faded exercise 2

### Assemble the generator Sequential

Complete `build_generator(hidden_channels)`. The reverse and pairing are done for you. Fill in the loop body that, for each `(in_c, out_c)` pair, appends the three generator modules to `blocks` in the correct order: `ConvTranspose2d` (kernel 4, stride 2, padding 1, no bias), then `BatchNorm2d`, then `ReLU(inplace=True)`.

**Fill in:** appending the ConvTranspose2d, BatchNorm2d, and ReLU modules for one (in_c, out_c) pair

In [ ]:
import torch.nn as nn

def build_generator(hidden_channels):
    gen_channels = hidden_channels[::-1]
    gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))
    blocks = []
    for in_c, out_c in gen_pairs:
        raise NotImplementedError()  # TODO: appending the ConvTranspose2d, BatchNorm2d, and ReLU modules for one (in_c, out_c) pair
    return nn.Sequential(*blocks)


def _test():
    import torch.nn as nn
    t.manual_seed(0)
    hidden = [32, 64, 128, 256]
    gen = build_generator(hidden)
    rev = hidden[::-1]
    n_pairs = len(rev) - 1
    assert len(gen) == 3 * n_pairs, len(gen)
    # module-type order per block
    for i in range(n_pairs):
        assert isinstance(gen[3*i], nn.ConvTranspose2d), i
        assert isinstance(gen[3*i+1], nn.BatchNorm2d), i
        assert isinstance(gen[3*i+2], nn.ReLU), i
    # first conv must enter at the reversed-first channel
    assert gen[0].in_channels == rev[0], gen[0].in_channels
    assert gen[0].bias is None
    # spatial doubling: input enters at rev[0] channels
    x = t.randn(2, rev[0], 4, 4)
    out = gen(x)
    assert tuple(out.shape) == (2, rev[-1], 4 * 2 ** n_pairs, 4 * 2 ** n_pairs), tuple(out.shape)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

def build_generator(hidden_channels):
    gen_channels = hidden_channels[::-1]
    gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))
    blocks = []
    for in_c, out_c in gen_pairs:
        blocks.append(nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False))
        blocks.append(nn.BatchNorm2d(out_c))
        blocks.append(nn.ReLU(inplace=True))
    return nn.Sequential(*blocks)
```
</details>